In [ ]:
import hft
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from scipy.stats import pearsonr, zscore, spearmanr

from hft.utils.wrapper import trade_to_depth
from hft.utils.validate import plot_stats
from hft.utils.target.mid_price_changes import all_return
from hft.utils.target import filled_return, mid_price_changes, mp_changes
from hft.utils.combine import linear_model, SGDlinear_model
from hft.utils.format import purged_train_test_split, single_split
from hft.utils.format import depth_to_depth


from hft.signal.arrive_rate import arrive_rate
from hft.signal.cofi import cofi
from hft.signal.depth_age import depth_bid_age, depth_ask_age
from hft.signal.depth_changes import depth_bid_change, depth_ask_change
from hft.signal.fair_spread import fair_spread
from hft.signal.large_jump import large_jump
from hft.signal.llt import llt
from hft.signal.oir import oir
from hft.signal.order_flow import oflow
from hft.signal.price_distance import price_distance
from hft.signal.price_impact import price_impact
from hft.signal.return_lag import ask_lag,bid_lag
from hft.signal.swr import swr
from hft.signal.tick_vpin import tick_vpin
from hft.signal.volume_at_same_price  import volume_at_same_price
from hft.signal.volume import ask_volume, bid_volume
from hft.signal.weakoir import weakoir
from hft.signal.wss import wss



In [ ]:
# 数据导入
#ob = pd.read_csv('/Users/wook/Desktop/因子挖掘/raw_data/BTCUSDT_book_snapshot_5_20240528.csv.gz')
#ob = pd.read_csv('/Users/wook/Desktop/因子挖掘/processed_data_10s/BTCUSDT_book_snapshot_5_20240528_10s_orderbook.csv')
ob = pd.read_csv('/Users/wook/Desktop/因子挖掘/Sample data/raw/BTC分段数据/book5_snapshot_1s/20250602_000_bear_alignment_0445_snapshot.csv')
tr = pd.read_csv('/Users/wook/Desktop/因子挖掘/Sample data/raw/BTC分段数据/trade/20250602_000_bear_alignment_0445_trade.csv')
#tr = pd.read_csv('/Users/wook/Desktop/因子挖掘/processed_data_1s/BTCUSDT_trades_20240528_1s_trades.csv')


# 对于ob
ob = ob.rename(columns={
    'timestamp': 'ts',
    'bids[0].price': 'bp1', 'bids[0].amount': 'bv1',
    'asks[0].price': 'ap1', 'asks[0].amount': 'av1',
    'bids[1].price': 'bp2', 'bids[1].amount': 'bv2',
    'asks[1].price': 'ap2', 'asks[1].amount': 'av2',
    'bids[2].price': 'bp3', 'bids[2].amount': 'bv3',
    'asks[2].price': 'ap3', 'asks[2].amount': 'av3',
    'bids[3].price': 'bp4', 'bids[3].amount': 'bv4',
    'asks[3].price': 'ap4', 'asks[3].amount': 'av4',
    'bids[4].price': 'bp5', 'bids[4].amount': 'bv5',
    'asks[4].price': 'ap5', 'asks[4].amount': 'av5'
})

tr = tr.rename(columns={'timestamp':'ts', 'price':'p', 'amount':'v'})
tr['v'] = np.where(tr['side'] == 'sell', -tr['v'], tr['v'])

In [ ]:
Fre = 600
# 计算所有因子
arrive_rate_factor = arrive_rate(
    datas={'depth5': ob, 'trade': tr},
    params={'n': 600}  #用逐笔交易数据，分析n笔成交的平均时间
)

# 公平价差因子
cofi_factor = cofi(
    datas={'depth5': ob},
    params={}
)

# 深度账簿年龄因子
depth_bid_age_factor = depth_bid_age(
    datas={'depth5': ob},
    params={'n': Fre}
)
depth_ask_age_factor = depth_ask_age(
    datas={'depth5': ob},
    params={'n': Fre}
)

# 深度账簿变化因子
depth_bid_change_factor = depth_bid_change(
    datas={'depth5': ob},
    params={'n': Fre}
)
depth_ask_change_factor = depth_ask_change(
    datas={'depth5': ob},
    params={'n': Fre}
)

# 公平价差因子
fair_spread_factor = fair_spread(
    datas={'depth5': ob},
    params={}   
)

# 大跳因子（向上和向下）
large_jump_up_factor = large_jump(
    datas={'depth5': ob},
    params={'n': Fre, 'direct': 'up', 'jump_ratio': 0.001}
)

large_jump_down_factor = large_jump(
    datas={'depth5': ob},
    params={'n': 100, 'direct': 'down', 'jump_ratio': 0.001}
)

# 计算LLT平滑值
llt_factor = llt(
    datas={"depth5": ob},
    params={"n": Fre }
)

# 订单失衡比率
oir_factor = oir(
    datas={'depth5': ob},
    params={}
)

# 订单流因子
bid_order_flow_factor = oflow(
    datas={'depth5': ob},
    params={'side': 'bid', 'bend_ratio': 4}
)
ask_order_flow_factor = oflow(
    datas={'depth5': ob},
    params={'side': 'ask', 'bend_ratio': 4}
)


# 价格距离因子
ask_price_distance_factor = price_distance(
    datas={'depth5': ob},
    params={'n': Fre, 'side': 'ask'}
)
bid_price_distance_factor = price_distance(
    datas={'depth5': ob},
    params={'n': Fre, 'side': 'bid'}
)

# 价格冲击因子
price_impact_factor = price_impact(
    datas={'depth5': ob},
    params={'n': 5}
    
)

# price_swap_factor = price_swap(
#     datas={'depth5': ob},
#     params={}
# )

#滞后因子
ask_lag_factor = ask_lag(
    datas={'depth5': ob},
    params={'n': Fre}
)
bid_lag_factor = bid_lag(
    datas={'depth5': ob},
    params={'n': Fre}
)

#ser因子
ask_swr_factor = swr( 
    datas={'depth5': ob},
     params={'side': 'ask'}
)
bid_swr_factor = swr( 
    datas={'depth5': ob},
     params={'side': 'bid'}
)
#tick_vpin因子
tick_vpin_factor = tick_vpin(
    datas={'depth5': ob, 'trade': tr},
    params={'n': 10}
)

#volume_at_same_price因子
volume_at_same_price_factor = volume_at_same_price(
    datas={'depth5': ob, 'trade': tr},
    params={'n': 600}
)


# 交易量因子
ask_volume_factor = ask_volume(
    datas={'depth5': ob},
    params={}
)
bid_volume_factor = bid_volume(
    datas={'depth5': ob},
    params={}
)

# 弱订单失衡比率
weakoir_factor = weakoir(
    datas={'depth5': ob},
    params={}
)

#wss因子
wss_factor = wss(
    datas={'depth5': ob},
    params={'n':5}
)


# 合并所有因子为DataFrame
factors_df = pd.DataFrame({
    'arrive_rate': arrive_rate_factor,
    'cofi': cofi_factor,
    'depth_bid_age': depth_bid_age_factor,
    'depth_ask_age': depth_ask_age_factor,
    'depth_bid_change': depth_bid_change_factor,
    'depth_ask_change': depth_ask_change_factor,
    'fair_spread': fair_spread_factor,
    'large_jump_up': large_jump_up_factor,
    'large_jump_down': large_jump_down_factor,
    'llt': llt_factor,
    'oir': oir_factor,
    'bid_order_flow': bid_order_flow_factor,
    'ask_order_flow': ask_order_flow_factor,
    'ask_price_distance': ask_price_distance_factor,
    'bid_price_distance': bid_price_distance_factor,
    'price_impact': price_impact_factor,
    'ask_lag': ask_lag_factor,
    'bid_lag': bid_lag_factor,
    'ask_swr': ask_swr_factor,
    'bid_swr': bid_swr_factor,
    'tick_vpin': tick_vpin_factor,
    'volume_at_same_price': volume_at_same_price_factor,
    'ask_volume': ask_volume_factor,
    'bid_volume': bid_volume_factor,
    'weakoir': weakoir_factor,
    'wss': wss_factor
    })

# 查看因子数据
print(factors_df.head())

# 计算因子的基本统计量
print(factors_df.describe())

# 可视化部分因子
plt.figure(figsize=(15, 10))

# 绘制前4个因子的时间序列图
for i, col in enumerate(factors_df.columns[:25]):
    plt.subplot(5, 5, i+1)
    plt.plot(factors_df[col])
    plt.title(col)
    plt.grid(True)

plt.tight_layout()
plt.show()



In [ ]:
from hft.utils.validate import plot_stats,MDI,numeric_stats,PFV

In [ ]:
# depth_bid_age
depth_bid_age_result = numeric_stats({"datas":{"orderbook":ob},"feature":depth_bid_age_factor})
depth_bid_age_result


In [ ]:
plot_stats({"datas":{"orderbook":ob},"feature":depth_bid_age_factor})

In [ ]:
# depth_ask_age
depth_ask_age_result = numeric_stats({"datas":{"orderbook":ob},"feature":depth_ask_age_factor})
depth_ask_age_result
plot_stats({"datas":{"orderbook":ob},"feature":depth_ask_age_factor})


In [ ]:
# depth_bid_change
depth_bid_change_result = numeric_stats({"datas":{"orderbook":ob},"feature":depth_bid_change_factor})
depth_bid_change_result
plot_stats({"datas":{"orderbook":ob},"feature":depth_bid_change_factor})


In [ ]:
# depth_ask_change
depth_ask_change_result = numeric_stats({"datas":{"orderbook":ob},"feature":depth_ask_change_factor})
depth_ask_change_result
plot_stats({"datas":{"orderbook":ob},"feature":depth_ask_change_factor})

In [ ]:
# fair_spread
fair_spread_result = numeric_stats({"datas":{"orderbook":ob},"feature":fair_spread_factor})
fair_spread_result
plot_stats({"datas":{"orderbook":ob},"feature":fair_spread_factor})


In [ ]:
# oir
oir_result = numeric_stats({"datas":{"orderbook":ob},"feature":oir_factor})
oir_result
plot_stats({"datas":{"orderbook":ob},"feature":oir_factor})


In [ ]:
# bid_order_flow
bid_order_flow_result = numeric_stats({"datas":{"orderbook":ob},"feature":bid_order_flow_factor})
bid_order_flow_result
plot_stats({"datas":{"orderbook":ob},"feature":bid_order_flow_factor})



In [ ]:
# ask_order_flow
ask_order_flow_result = numeric_stats({"datas":{"orderbook":ob},"feature":ask_order_flow_factor})
ask_order_flow_result
plot_stats({"datas":{"orderbook":ob},"feature":ask_order_flow_factor})



In [ ]:
# ask_price_distance
ask_price_distance_result = numeric_stats({"datas":{"orderbook":ob},"feature":ask_price_distance_factor})
ask_price_distance_result
plot_stats({"datas":{"orderbook":ob},"feature":ask_price_distance_factor})



In [ ]:
# bid_price_distance
bid_price_distance_result = numeric_stats({"datas":{"orderbook":ob},"feature":bid_price_distance_factor})
bid_price_distance_result
plot_stats({"datas":{"orderbook":ob},"feature":bid_price_distance_factor})


In [ ]:
# price_impact
price_impact_result = numeric_stats({"datas":{"orderbook":ob},"feature":price_impact_factor})
price_impact_result
plot_stats({"datas":{"orderbook":ob},"feature":price_impact_factor})



In [ ]:
# weakoir
weakoir_result = numeric_stats({"datas":{"orderbook":ob},"feature":weakoir_factor})
weakoir_result
plot_stats({"datas":{"orderbook":ob},"feature":weakoir_factor})



In [ ]:
# ask_lag
ask_lag_result = numeric_stats({"datas":{"orderbook":ob},"feature":ask_lag_factor})
ask_lag_result
plot_stats({"datas":{"orderbook":ob},"feature":ask_lag_factor})


In [ ]:
# bid_lag
bid_lag_result = numeric_stats({"datas":{"orderbook":ob},"feature":bid_lag_factor})
bid_lag_result
plot_stats({"datas":{"orderbook":ob},"feature":bid_lag_factor})

In [ ]:
# ask_swr
ask_swr_result = numeric_stats({"datas":{"orderbook":ob},"feature":ask_swr_factor})
ask_swr_result
plot_stats({"datas":{"orderbook":ob},"feature":ask_swr_factor})



In [ ]:
# bid_swr
bid_swr_result = numeric_stats({"datas":{"orderbook":ob},"feature":bid_swr_factor})
bid_swr_result
plot_stats({"datas":{"orderbook":ob},"feature":bid_swr_factor})

In [ ]:
# tick_vpin
tick_vpin_result = numeric_stats({"datas":{"orderbook":ob},"feature":tick_vpin_factor})
tick_vpin_result
plot_stats({"datas":{"orderbook":ob},"feature":tick_vpin_factor})


In [ ]:
# ask_volume_at_same_price
ask_volume_at_same_price_result = numeric_stats({"datas":{"orderbook":ob},"feature":volume_at_same_price_factor})
ask_volume_at_same_price_result
plot_stats({"datas":{"orderbook":ob},"feature":volume_at_same_price_factor})


In [ ]:
# ask_volume
ask_volume_result = numeric_stats({"datas":{"orderbook":ob},"feature":ask_volume_factor})
ask_volume_result
plot_stats({"datas":{"orderbook":ob},"feature":ask_volume_factor})

In [ ]:
# bid_volume
bid_volume_result = numeric_stats({"datas":{"orderbook":ob},"feature":bid_volume_factor})
bid_volume_result
plot_stats({"datas":{"orderbook":ob},"feature":bid_volume_factor})


In [ ]:
# wss
wss_result = numeric_stats({"datas":{"orderbook":ob},"feature":wss_factor})
wss_result
plot_stats({"datas":{"orderbook":ob},"feature":wss_factor})